# TRAITEMENT DES DONNES FUSIONNEES DE LA PREMIERE FEUILLE EXCEL

**Objectif** : Traitement/apurement des variables démographiques
* Date de naissance 
* Sexe
* Situation Matrimoniale
* Nombre d'enfants

Nous aurons également pour mission de créer de nouvelles variables à partir des informations disponibles. Il s'agit du statut dans l'emploi (Fonctionnaire ou contractuelle), l'âge, l'age de première prise de service, l'ancienneté dans l'organisme, nombre d'année d'expérience.

Un contrôle sera effectué sur le rattachement administratif:

* Matricule
* Fonction
* Services
* Organismes
* Situation dans l'emploi
* Emplois
* Fonction
* Classes/Echelons
* Lieu d'affectation

## Chargement des packages necessaires

In [50]:
pip install fastparquet

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [51]:
# Paramètres
import io
import pandas as pd
import boto3
import io
import unicodedata
import re
import numpy as np
import unicodedata
import unicodedata, re
import re

# FONCTIONS NECESSAIRES

### CREATION DE LA VARIABLE CATEGORIE 1 ET 2 

In [3]:
def build_categories(
    df: pd.DataFrame,
    grade_col="GRADE",
    grade2_col="GRADE 2",
    compute_categorie2=True,     # calcule CATEGORIE_2 depuis GRADE 2 si la colonne existe
    overwrite_categorie2=True    # réécrit CATEGORIE_2 même si déjà présente
):
    """
    - CATEGORIE_1 depuis GRADE (texte):
        'Non Fonctionnaire' si le texte contient 'non fonctionnaire' (insensible à la casse/espaces)
        sinon extrait la lettre A-D depuis 'Catégorie A7', 'Catégorie B', etc.
        NaN -> 'Non Fonctionnaire'
    - CATEGORIE_2 depuis GRADE 2 (lettres A-D):
        'A7' -> 'A', 'B' -> 'B', autres -> NaN -> 'Non Fonctionnaire'
    - Les 2 colonnes sont typées 'category' avec les mêmes modalités ordonnées.

    Retourne: df (copie) avec CATEGORIE_1 (et CATEGORIE_2 si demandé).
    """

    out = df.copy()
    cat_order = ["Non Fonctionnaire", "A", "B", "C", "D"]

    # ---------- helpers ----------
    def _cat_from_grade2_letter(val):
        """ Cette fonction transforme un GRADE 2 en catégorie """
        if pd.isna(val):
            return pd.NA  # si valeur manquante, on retourne pd.NA.
        s = str(val).strip().upper()  # on convertit en chaîne, on enlève les espaces et on met en majuscules
        if re.fullmatch(r"[ABCD]\d+", s):   # A7, B3, C1, D2...
            return s[0]
        if re.fullmatch(r"[ABCD]", s):      # A, B, C, D
            return s
        return pd.NA

    def _parse_cat_from_grade_text(val):
        if pd.isna(val):
            return pd.NA
        t = str(val)
        # ex.: "Non   fonctionnaire", "non-fonctionnaire"
        if re.search(r"\bnon\s*fonctionnaire\b", t, flags=re.I):
            return "Non Fonctionnaire"
        # ex.: "Catégorie A", "Categorie B7", "catégorie c titulaire"
        m = re.search(r"Cat[ée]gorie\s+([A-D])(?:\s*\d+)?", t, flags=re.I)
        if m:
            return m.group(1).upper()
        return pd.NA

    # ---------- CATEGORIE_1 depuis GRADE ----------
    if grade_col not in out.columns:
        raise KeyError(f"Colonne '{grade_col}' absente.")
    out["CATEGORIE_1"] = out[grade_col].apply(_parse_cat_from_grade_text)
    out["CATEGORIE_1"] = out["CATEGORIE_1"].fillna("Non Fonctionnaire") # Remplace les NaN par "Non Fonctionnaire"
    out["CATEGORIE_1"] = pd.Categorical(out["CATEGORIE_1"], categories=cat_order, ordered=True) #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    # ---------- (optionnel) CATEGORIE_2 depuis GRADE 2 ----------
    if compute_categorie2 and (grade2_col in out.columns):
        if overwrite_categorie2 or ("CATEGORIE_2" not in out.columns):
            cat2_raw = out[grade2_col].apply(_cat_from_grade2_letter)
            cat2 = cat2_raw.astype(object)
            cat2[pd.isna(cat2)] = "Non Fonctionnaire" # Remplace les NaN par "Non Fonctionnaire"
            out["CATEGORIE_2"] = pd.Categorical(cat2, categories=cat_order, ordered=True) # #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    return out 

### CREATION DE LA VARIABLE STATUT

In [4]:
def build_statut_from_cats(df: pd.DataFrame, 
                           emploi_col: str = "EMPLOI_NORM",
                           cat1_col: str | None = None,
                           cat1_candidates: tuple[str,str] = ("CATEGORIE_1", "CATEGORIE 1"),
                           cat2_col: str | None = None,
                           cat2_candidates: tuple[str,str] = ("CATEGORIE_2", "CATEGORIE 2"),
                           label_cas: str = "Cas particulier"):

    import unicodedata, re
    import numpy as np
    import pandas as pd

    out = df.copy()

    # --- Trouver les colonnes ---
    def _pick_col(cand_list):
        for c in cand_list:
            if c in out.columns:
                return c
        return None

    if cat1_col is None:
        cat1_col = _pick_col(cat1_candidates)
    if cat2_col is None:
        cat2_col = _pick_col(cat2_candidates)

    for col, name in zip([cat1_col, cat2_col, emploi_col], ["CATEGORIE_1","CATEGORIE_2","EMPLOI_NORM"]):
        if col is None or col not in out.columns:
            raise KeyError(f"Colonne '{name}' introuvable.")

    # --- EMPLOI renseigné ---
    emp_norm = out[emploi_col].fillna("").astype(str).str.strip()
    has_emploi = emp_norm.ne("")

    # --- Normalisation catégories ---
    def _norm_ascii_lower(x):
        if pd.isna(x):
            return ""
        s = str(x).strip()
        s = unicodedata.normalize("NFKD", s)
        s = s.encode("ascii", "ignore").decode("ascii")
        s = re.sub(r"\s+", " ", s).strip().lower()
        return s

    c1_norm = out[cat1_col].map(_norm_ascii_lower)
    c2_norm = out[cat2_col].map(_norm_ascii_lower)

    c1_is_nf = c1_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)
    c2_is_nf = c2_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)

    c1_is_abcd = c1_norm.str.fullmatch(r"[abcd]", na=False)
    c2_is_abcd = c2_norm.str.fullmatch(r"[abcd]", na=False)

    is_nf_any = c1_is_nf | c2_is_nf
    is_nf_all = c1_is_nf & c2_is_nf
    is_abcd_any = c1_is_abcd | c2_is_abcd
    is_abcd_all = c1_is_abcd & c2_is_abcd

    # --- STATUT ---
    statut = np.select(
        [(has_emploi & is_nf_any) | (~has_emploi & is_abcd_any),
         has_emploi & is_abcd_all,
         ~has_emploi & is_nf_all],
        [label_cas, "Fonctionnaire", "Non Fonctionnaire"],
        default="Non Fonctionnaire"
    ).astype(object)

    out["STATUT"] = pd.Categorical(
        statut,
        categories=["Non Fonctionnaire", "Fonctionnaire", label_cas],
        ordered=True
    )

    # --- Rapport automatique ---
    rep = {
        "repartition_statut": out["STATUT"].value_counts().reindex(
            ["Non Fonctionnaire","Fonctionnaire",label_cas], fill_value=0
        )
    }

    return out, rep

### CREATION DE LA VARIABLE STATUT DEFINITIF 

Créer une variable Statut_def qui renseigne le statut définitif de l'individu sur une période donnée. En effet, un individu peut être fonctionnaire sur un organisme et ne pas contractuel sur un autre organisme. Le matricule permet d'identifier de manière unique un individu. On combinant le matricule et le code organisme on peut retrouver le statut de l'individu sur chaque organisme où il intervient. Pour une période de collecte donnée (mois et année), le statut définitif sera qu'il est fonctionnaire si sur au moins un organisme l'individu est fonctionnaire sinon non fonctionnaire. Attention, le contrôle se fait sur la période. Ici nous avons 72 périodes (janvier 2015 à décembre 2020).

In [5]:
def build_statut_def_periode(
    df: pd.DataFrame,
    statut_col: str = "STATUT",
    matricule_col: str = "MATRICULE",
    periode_col: str = "PERIODE",
    output_col: str = "Statut_def_mois",
    return_report: bool = True
):

    """
        Règle:
      - Pour chaque groupe (MATRICULE et PERIODE ), on définit le statut final:
          * Si au moins une ligne = 'Fonctionnaire' → Statut_def = 'Fonctionnaire'
          * Sinon si au moins une ligne = 'Cas particulier' → Statut_def = 'Cas particulier'
          * Sinon → 'Non Fonctionnaire'
          
          Colonnes attendues: statut_col, matricule_col, periode_col
    """
    out = df.copy()

    # Vérifications
    for col in [statut_col, matricule_col, periode_col]:
        if col not in out.columns:
            raise KeyError(f"Colonne '{col}' introuvable.")

    # Normalisation
    stat = out[statut_col].astype(str).str.strip().str.casefold()
    est_fonctionnaire = stat == "fonctionnaire"
    est_cas_particulier = stat == "cas particulier"

    # Agrégation par groupe. On regroupe par MATRICULE et PERIODE pour vérifier si au moins
    # une ligne est “Fonctionnaire” ou “Cas particulier”.
    key_cols = [matricule_col, periode_col]
    any_fonct = est_fonctionnaire.groupby([out[c] for c in key_cols]).transform("any")
    any_cas = est_cas_particulier.groupby([out[c] for c in key_cols]).transform("any")

    # Priorité pour le statut final
    out[output_col] = np.select(
        [any_fonct, any_cas],
        ["Fonctionnaire", "Cas particulier"],
        default="Non Fonctionnaire"
    )

    out[output_col] = pd.Categorical(
        out[output_col],
        categories=["Non Fonctionnaire", "Cas particulier", "Fonctionnaire"],
        ordered=True
    )

    if return_report:
        grp_summary = out.groupby(key_cols)[output_col].first()
        rep = {
            "Statut définifitif": out[output_col].value_counts()
                                   .reindex(["Non Fonctionnaire","Cas particulier","Fonctionnaire"], fill_value=0),
        }
        return out, rep

    return out

In [6]:
def build_statut_def_annee(df, 
                            statut_col="STATUT", 
                            matricule_col="MATRICULE",
                            date_collecte_col="DATE_COLLECTE",
                            output_col="Statut_def_annee",
                            return_report=True):
    """
    Détermine le statut définitif par MATRICULE et ANNEE (extraite de DATE_COLLECTE).
    
    Règle:
      - Si au moins une ligne = 'Fonctionnaire' → Statut_def = 'Fonctionnaire'
      - Sinon si au moins une ligne = 'Cas particulier' → Statut_def = 'Cas particulier'
      - Sinon → 'Non Fonctionnaire'
    
    Retour:
        - df mis à jour
        - report (distribution par Statut_def) si return_report=True
    """
    out = df.copy()

    # Vérifications colonnes
    for col in [statut_col, matricule_col, date_collecte_col]:
        if col not in out.columns:
            raise KeyError(f"Colonne '{col}' introuvable.")

    # Extraire l'année
    out[date_collecte_col] = pd.to_datetime(out[date_collecte_col], errors="coerce")
    out["ANNEE"] = out[date_collecte_col].dt.year

    # Normalisation du statut
    stat = out[statut_col].astype(str).str.strip().str.casefold()
    est_fonctionnaire = stat == "fonctionnaire"
    est_cas_particulier = stat == "cas particulier"

    # Regroupement par matricule et année
    key_cols = [matricule_col, "ANNEE"]
    any_fonct = est_fonctionnaire.groupby([out[c] for c in key_cols]).transform("any")
    any_cas = est_cas_particulier.groupby([out[c] for c in key_cols]).transform("any")

    # Détermination du statut définitif
    out[output_col] = np.select(
        [any_fonct, any_cas],
        ["Fonctionnaire", "Cas particulier"],
        default="Non Fonctionnaire"
    )

    out[output_col] = pd.Categorical(
        out[output_col],
        categories=["Non Fonctionnaire", "Cas particulier", "Fonctionnaire"],
        ordered=True
    )

    if return_report:
        rep = {
            "Statut_def_distribution": out[output_col].value_counts()
                                   .reindex(["Non Fonctionnaire","Cas particulier","Fonctionnaire"], fill_value=0)
        }
        return out, rep

    return out


### CREATION DE LA VARIABLE SEXE IMPUTE

In [7]:
def imputer_sexe(
    df: pd.DataFrame,
    sex_col="SEXE",               # Colonne brute contenant le sexe
    collect_col="DATE_COLLECTE",  # Colonne de date de collecte
    sex_valid_col="SEXE_IMPUTE",  # Colonne qui sera créée pour stocker les valeurs imputées
    debug=True                     # Affichage automatique par défaut
):
    """
    Impute les valeurs manquantes de sexe en utilisant le mode calculé
    au niveau ANNEE_COLLECTE × MOIS_COLLECTE.

    Paramètres :
        df : pd.DataFrame
            DataFrame à traiter
        sex_col : str
            Colonne brute du sexe
        collect_col : str
            Colonne de date de collecte
        sex_valid_col : str
            Nom de la colonne imputée à créer
        debug : bool
            Si True, affichage automatique des statistiques avant/après

    Retour :
        df : DataFrame avec la colonne imputée
        report : dictionnaire avec tableaux de statistiques
    """
    import pandas as pd
    import numpy as np

    # ---- Travail sur une copie pour ne pas modifier l'original ----
    df = df.copy()

    # ---- Conversion de la colonne date ----
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")

    # ---- Création des colonnes année et mois de collecte ----
    df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    df["MOIS_COLLECTE"] = df[collect_col].dt.month

    # ---- Initialisation de la colonne imputée avec les valeurs existantes ----
    df[sex_valid_col] = df[sex_col]

    # ---- Calcul du mode du sexe par groupe (année × mois) ----
    mode_sexe_groupes = (
        df.dropna(subset=[sex_col])  # On ignore les NaN
          .groupby(["ANNEE_COLLECTE", "MOIS_COLLECTE"])[sex_col]
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )

    # ---- Imputation : remplacer NaN par le mode du groupe ----
    df[sex_valid_col] = df.apply(
        lambda row: mode_sexe_groupes.get(
            (row["ANNEE_COLLECTE"], row["MOIS_COLLECTE"]), row[sex_valid_col]
        ) if pd.isna(row[sex_valid_col]) else row[sex_valid_col],
        axis=1
    )

    # ---- Statistiques avant/après imputation ----
    tab_sexe_avant = df[sex_col].value_counts(dropna=False).sort_index()
    tab_sexe_apres = df[sex_valid_col].value_counts(dropna=False).sort_index()
    tab_sexe_apres_pct = df[sex_valid_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    # ---- Tableau croisé avant/après ----
    crosstab_sexe = pd.crosstab(
        df[sex_col],
        df[sex_valid_col],
        dropna=False,
        margins=True,
        margins_name="Total"
    )

    # ---- Création du dictionnaire de report ----
    report = {
        "tab_sexe_avant": tab_sexe_avant,
        "tab_sexe_apres": tab_sexe_apres,
        "tab_sexe_apres_pct": tab_sexe_apres_pct,
        "crosstab_sexe": crosstab_sexe
    }

    # ---- Affichage automatique si debug=True ----
    if debug:
        print("=== Statistiques avant imputation ===")
        display(tab_sexe_avant.to_frame("Effectif"))
        print("\n=== Statistiques après imputation ===")
        display(tab_sexe_apres.to_frame("Effectif"))
        print("\n=== Pourcentages après imputation ===")
        display(tab_sexe_apres_pct.to_frame("Pourcentage (%)"))
        print("\n=== Tableau croisé avant/après ===")
        display(crosstab_sexe)

    return df, report

In [8]:
def verifier_variation_sexe(df, id_col="MATRICULE", sex_col="SEXE_IMPUTE", collect_col="DATE_COLLECTE"):
    """
    Vérifie si le sexe d'un individu varie dans le temps et met à jour SEXE_IMPUTE
    avec la valeur la plus récente selon DATE_COLLECTE.

    Retourne :
      - variation : DataFrame des individus pour lesquels le sexe n'est pas constant
      - details : DataFrame détaillé de ces individus
      - df : DataFrame mis à jour avec SEXE_IMPUTE actualisé
      - stats : dictionnaire avec tableaux avant/après
    """
    df = df.copy()
    
    # Conversion en datetime si nécessaire
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")
    
    # Tableau avant
    tab_avant = df[sex_col].value_counts(dropna=False).sort_index()
    
    # Nombre de valeurs distinctes de sexe par individu
    variation = df.groupby(id_col)[sex_col].nunique().reset_index()
    variation = variation.rename(columns={sex_col: "nb_valeurs_distinctes"})
    
    # Sélection des individus avec plus d'une valeur distincte
    variation = variation[variation["nb_valeurs_distinctes"] > 1]
    
    # Détails pour inspection
    details = df[df[id_col].isin(variation[id_col])].sort_values([id_col, collect_col])
    
    print(f"Nombre d'individus dont le sexe varie dans le temps : {variation.shape[0]}")

    # Mettre à jour SEXE_IMPUTE : prendre la valeur la plus récente
    latest_sex = df.sort_values([id_col, collect_col]).groupby(id_col, as_index=False).last()[[id_col, sex_col]]
    df = df.drop(columns=[sex_col]).merge(latest_sex, on=id_col, how="left")

    # Tableau après
    tab_apres = df[sex_col].value_counts(dropna=False).sort_index()
    tab_apres_pct = df[sex_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    stats = {
        "tab_avant": tab_avant,
        "tab_apres": tab_apres,
        "tab_apres_pct": tab_apres_pct
    }

    return variation, details, df, stats


### CREATION DE LA VARIABLE AGE IMPUTE

In [9]:
def build_age_valide(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    birth_col="DATE NAISSANCE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",                  # ⚡️ ajouter le sexe
    age_min=18, age_max=70,
    do_impute_age=True
):
    """
    Construit les variables d'âge à partir des dates de naissance.
    Toujours : on prend la date la plus récente pour chaque matricule.

    Étapes :
      1) Nettoyage dates collecte et naissance
      2) Correction date naissance → la plus récente disponible par matricule
      3) Calcul AGE, AGE_VALIDE
      4) Option : imputation AGE_VALIDE manquant → AGE_IMPUTE
        (imputation par médiane par année, mois et sexe)
    """

    df = df.copy()

    # -------- Utilitaires --------
    def to_datetime_mixed(s):
        try:
            return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
        except TypeError:
            return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

    def clean_birthdates(series):
        s = series.astype(str).str.strip().str.lower()
        time_only = s.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?")
        zero_date = s.str.fullmatch(r"(0{1,2}[/\-]0{1,2}[/\-]0{2,4}|0000-00-00)")
        empties   = s.isin(["", "nan", "nat", "none", "nul", "null"])
        s = series.mask(time_only | zero_date | empties, np.nan)
        dt = to_datetime_mixed(s)

        # Numéros Excel
        serial = pd.to_numeric(series, errors="coerce")
        serial_mask = dt.isna() & serial.notna() & serial.between(1, 60000)
        if serial_mask.any():
            dt_serial = pd.to_datetime(serial[serial_mask], unit="D", origin="1899-12-30", errors="coerce")
            dt.loc[serial_mask] = dt_serial
        return dt

    # -------- 0) DATE_COLLECTE --------
    df[collect_col] = to_datetime_mixed(df[collect_col])
    if "ANNEE_COLLECTE" not in df.columns:
        df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    if "MOIS_COLLECTE" not in df.columns:
        df["MOIS_COLLECTE"] = df[collect_col].dt.month

    # -------- 1) Nettoyage DATE NAISSANCE --------
    df["DATE_NAISSANCE_CLEAN"] = clean_birthdates(df[birth_col])
    df["ANNEE_NAISSANCE"] = df["DATE_NAISSANCE_CLEAN"].dt.year
    df["MOIS_NAISSANCE"]  = df["DATE_NAISSANCE_CLEAN"].dt.month
    df["JOUR_NAISSANCE"]  = df["DATE_NAISSANCE_CLEAN"].dt.day

    # -------- 2) DATE_NAISSANCE_CORRIGEE = date la + récente --------
    tmp = (
        df.dropna(subset=["DATE_NAISSANCE_CLEAN"])
          .sort_values([matricule_col, collect_col], ascending=[True, False])
          .drop_duplicates(subset=[matricule_col], keep="first")
          [[matricule_col, "DATE_NAISSANCE_CLEAN"]]
          .rename(columns={"DATE_NAISSANCE_CLEAN": "DATE_NAISSANCE_CORRIGEE"})
    )
    df = df.drop(columns=["DATE_NAISSANCE_CORRIGEE"], errors="ignore")
    df = df.merge(tmp, on=matricule_col, how="left", validate="many_to_one")
    df["DATE_NAISSANCE_CORRIGEE"] = pd.to_datetime(df["DATE_NAISSANCE_CORRIGEE"], errors="coerce")
    df["ANNEE_NAISSANCE_CORRIGEE"] = df["DATE_NAISSANCE_CORRIGEE"].dt.year
    df["MOIS_NAISSANCE_CORRIGEE"]  = df["DATE_NAISSANCE_CORRIGEE"].dt.month
    df["JOUR_NAISSANCE_CORRIGEE"]  = df["DATE_NAISSANCE_CORRIGEE"].dt.day

    # -------- 3) Calcul AGE --------
    birth = df["DATE_NAISSANCE_CORRIGEE"]
    ref   = df[collect_col]
    age = pd.Series(pd.NA, index=df.index, dtype="Float64")
    mask = birth.notna() & ref.notna()
    if mask.any():
        ydiff = ref[mask].dt.year - birth[mask].dt.year
        before_bday = (ref[mask].dt.month < birth[mask].dt.month) | (
            (ref[mask].dt.month == birth[mask].dt.month) & (ref[mask].dt.day < birth[mask].dt.day)
        )
        age.loc[mask] = (ydiff - before_bday.astype(int)).astype("Float64")
    df["AGE"] = age

    # -------- 4) AGE_VALIDE --------
    df["AGE_VALIDE"] = df["AGE"].where((df["AGE"].ge(age_min)) & (df["AGE"].le(age_max)))

    # -------- 5) AGE_IMPUTE --------
    if do_impute_age:
        # clés : année, mois et sexe
        group_keys = ["ANNEE_COLLECTE", "MOIS_COLLECTE"]
        if sex_col in df.columns:
            group_keys.append(sex_col)

        med = df.groupby(group_keys)["AGE_VALIDE"].transform("median")
        global_med = df["AGE_VALIDE"].median()
        df["AGE_IMPUTE"] = df["AGE_VALIDE"].where(df["AGE_VALIDE"].notna(), med.fillna(global_med)).round(0).astype("Int64")

    return df

### CREATION DE LA VARIABLE AGE DE PRISE DE SERVICE 


In [10]:
def build_age_service(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    year_collect_col="ANNEE_COLLECTE",
    month_collect_col="MOIS_COLLECTE",
    recompute_corrected=True,
    age_min=18,
    age_max=40,
    days_per_year=365,
    return_tables=True,
    sample_anomalies=10
):
    import pandas as pd
    import numpy as np

    df = df.copy()

    # ---------- utilitaires ----------
    def _to_datetime_mixed(s):
        try:
            return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
        except TypeError:
            return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

    def _clean_dates_generic(series):
        s = series.copy()
        s_str = s.astype(str).str.strip().str.lower()
        time_only = s_str.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?")
        zero_date = s_str.str.fullmatch(r"(0{1,2}[/\-]0{1,2}[/\-]0{2,4}|0000-00-00)")
        empties   = s_str.isin(["", "nan", "nat", "none", "nul", "null"])
        s = s.mask(time_only | zero_date | empties, np.nan)
        dt = _to_datetime_mixed(s)
        serial = pd.to_numeric(s_str, errors="coerce")
        serial_mask = dt.isna() & serial.notna() & serial.between(1, 60000)
        if serial_mask.any():
            dt_serial = pd.to_datetime(serial[serial_mask], unit="D",
                                       origin="1899-12-30", errors="coerce")
            dt.loc[serial_mask] = dt_serial
        return dt

    # ---------- 0) Sécuriser DATE_COLLECTE et dérivées ----------
    df[collect_col] = _to_datetime_mixed(df[collect_col])
    if year_collect_col not in df.columns:
        df[year_collect_col] = df[collect_col].dt.year
    if month_collect_col not in df.columns:
        df[month_collect_col] = df[collect_col].dt.month

    # ---------- 1) Nettoyage PRISE DE SERVICE ----------
    df["PRISE_DE_SERVICE_CLEAN"] = _clean_dates_generic(df[start_col_raw])

    # ---------- 2) Diagnostics ----------
    nb_dates_par_matricule = df.groupby(matricule_col)["PRISE_DE_SERVICE_CLEAN"].nunique(dropna=True)
    matricules_problemes = nb_dates_par_matricule[nb_dates_par_matricule > 1].index.tolist()
    anomalies_dates = (
        df[df[matricule_col].isin(matricules_problemes)]
        [[matricule_col, collect_col, "PRISE_DE_SERVICE_CLEAN"]]
        .sort_values([matricule_col, collect_col])
    )

    # ---------- 3) PRISE_DE_SERVICE_CORRIGEE ----------
    if recompute_corrected or (start_col_corrected not in df.columns):
        panel_sorted = df.sort_values([matricule_col, collect_col], ascending=[True, False])
        ps_corr = (
            panel_sorted
            .drop_duplicates(subset=[matricule_col], keep="first")
            [[matricule_col, "PRISE_DE_SERVICE_CLEAN"]]
            .rename(columns={"PRISE_DE_SERVICE_CLEAN": start_col_corrected})
        )
        if start_col_corrected in df.columns:
            df.drop(columns=[start_col_corrected], inplace=True)
        df = df.merge(ps_corr, on=matricule_col, how="left", validate="many_to_one")

    df[start_col_corrected] = _to_datetime_mixed(df[start_col_corrected])

    # ---------- 4) AGE_AU_SERVICE ----------
    # → Créer dès le départ en Float64 (évite dtype=object plus tard)
    df["AGE_AU_SERVICE"] = pd.Series(pd.NA, index=df.index, dtype="Float64")
    df["AGE_AU_SERVICE_VALIDE"] = pd.Series(pd.NA, index=df.index, dtype="Float64")

    # S'assurer que AGE_IMPUTE est numérique
    if "AGE_IMPUTE" in df.columns:
        df["AGE_IMPUTE"] = pd.to_numeric(df["AGE_IMPUTE"], errors="coerce").astype("Float64")

    mask = df["AGE_IMPUTE"].notna() & df[start_col_corrected].notna()
    if mask.any():
        delta_days = (df.loc[mask, collect_col] - df.loc[mask, start_col_corrected]).dt.days
        # conversion en années ~ (jours // 365)
        df.loc[mask, "AGE_AU_SERVICE"] = (df.loc[mask, "AGE_IMPUTE"] - (delta_days // days_per_year)).astype("Float64")

    # Bornage en Float64
    ok = df["AGE_AU_SERVICE"].ge(age_min) & df["AGE_AU_SERVICE"].le(age_max)
    df.loc[ok, "AGE_AU_SERVICE_VALIDE"] = df.loc[ok, "AGE_AU_SERVICE"].astype("Float64")

    # ---------- 5) Imputation (médiane ANNEE×MOIS×SEXE_IMPUTE) ----------
    group_keys = [year_collect_col, month_collect_col, sex_col]
    med_group = df.groupby(group_keys)["AGE_AU_SERVICE_VALIDE"].transform("median")
    global_med = df["AGE_AU_SERVICE_VALIDE"].median()

    # Forcer numérique avant round
    impute_base = df["AGE_AU_SERVICE_VALIDE"].where(
        df["AGE_AU_SERVICE_VALIDE"].notna(), med_group.fillna(global_med)
    )
    impute_base = pd.to_numeric(impute_base, errors="coerce").astype("Float64")

    df["AGE_AU_SERVICE_IMPUTE"] = impute_base.round(0).astype("Float64")

    # ---------- 6) Report / tableaux ----------
    if return_tables:
        tab_valide = df["AGE_AU_SERVICE_VALIDE"].value_counts(dropna=False).sort_index()
        tab_valide_pct = df["AGE_AU_SERVICE_VALIDE"].value_counts(normalize=True, dropna=False).sort_index()
        tab_impute = df["AGE_AU_SERVICE_IMPUTE"].value_counts(dropna=False).sort_index()
        tab_impute_pct = df["AGE_AU_SERVICE_IMPUTE"].value_counts(normalize=True, dropna=False).sort_index()

        report = {
            "n_matricules_multi_service": len(matricules_problemes),
            "anomalies_dates_head": anomalies_dates.head(sample_anomalies),
            "tab_age_service_valide": pd.DataFrame({
                "Effectif": tab_valide,
                "Pourcentage (%)": (tab_valide_pct * 100).round(2)
            }),
            "tab_age_service_impute": pd.DataFrame({
                "Effectif": tab_impute,
                "Pourcentage (%)": (tab_impute_pct * 100).round(2)
            }),
        }
        return df, report

    return df


### CREATION DE LA VARIBALE ANCIENNTE DANS L'ORGANISME ET DEPUIS LA PRISE DE FONCTION

#### CREATION DE LA VARIBALE ANCIENNTE  DEPUIS LA PRISE DE FONCTION

In [11]:
def build_anciennete_mensuelle(
    df: pd.DataFrame,
    matricule_col: str = "MATRICULE",
    org_col: str | None = "CODE_ORGANISME",   # mettre None pour ignorer l’organisme
    collect_col: str = "DATE_COLLECTE",
    start_service_col: str = "PRISE_DE_SERVICE_CORRIGEE",  # ne change pas d’une année à l’autre

    # Fenêtre d’observation (ex. 2015–2020)
    min_period: str = "2015-01",   # inclus
    max_period: str = "2020-12",   # inclus

    # Options de sortie
    return_by_org: bool = True,    # ancienneté par organisme si org_col != None
    return_by_person: bool = True, # ancienneté globale par personne (tous organismes)
    keep_counts: bool = True,      # garder NB_APPA_MOIS_* et PRESENCE_MOIS_*

    # Pré-compte avant la fenêtre (si prise de service antérieure à min_period)
    include_prewindow_base: bool = True
):
    """
    Calcule l'ancienneté en MOIS :
      - avant la fenêtre : si include_prewindow_base=True et prise de service < min_period,
        on ajoute une base d'ancienneté continue du mois de prise de service jusqu'au mois précédent min_period.
      - dans la fenêtre [min_period, max_period] : on additionne UNIQUEMENT les mois effectivement observés
        (présence = au moins une ligne dans le mois).

    Sorties dans le DataFrame retourné (jointes au niveau mensuel) :
      - Par PERSONNE : NB_APPA_MOIS_PERS, PRESENCE_MOIS_PERS, ANCIENNETE_MOIS_PERS,
        BASE_PRE2015_MOIS_PERS (si include_prewindow_base),
        ANCIENNETE_MOIS_PERS_TOTAL (= base pré-fenêtre + observé),
        ANCIENNETE_AN_PERS, ANCIENNETE_AN_PERS_TOTAL.
      - Par ORGANISME : NB_APPA_MOIS_ORG, PRESENCE_MOIS_ORG, ANCIENNETE_MOIS_ORG, ANCIENNETE_AN_ORG.
    """
    out = df.copy()

    # ---------- 0) Dates et borne de fenêtre ----------
    out[collect_col] = pd.to_datetime(out[collect_col], errors="coerce")
    if out[collect_col].isna().all():
        raise ValueError(f"Toutes les valeurs de '{collect_col}' sont manquantes ou invalides.")

    # PERIODE = début de mois (clé mensuelle)
    out["PERIODE"] = out[collect_col].dt.to_period("M").dt.to_timestamp(how="start")

    # Fenêtre [min_period, max_period] (début de mois)
    min_p = pd.Period(min_period, freq="M").to_timestamp(how="start")
    max_p = pd.Period(max_period, freq="M").to_timestamp(how="start")

    # Filtre fenêtre
    out = out.loc[(out["PERIODE"] >= min_p) & (out["PERIODE"] <= max_p)].copy()

    # Calendrier mensuel complet pour la fenêtre
    full_months = pd.date_range(min_p, max_p, freq="MS")  # MonthStart (DatetimeIndex)

    # ---------- 1) Prise de service fixe (une par matricule) ----------
    if start_service_col in out.columns:
        tmp_ps = (
            out[[matricule_col, start_service_col]]
            .dropna()
            .sort_values([matricule_col, start_service_col])  # la plus ANCIENNE
            .drop_duplicates(subset=[matricule_col], keep="first")
            .rename(columns={start_service_col: "PRISE_DE_SERVICE_FIXE"})
        )
        out = out.drop(columns=["PRISE_DE_SERVICE_FIXE"], errors="ignore")
        out = out.merge(tmp_ps, on=matricule_col, how="left")
    else:
        out["PRISE_DE_SERVICE_FIXE"] = pd.NaT

    out["PRISE_DE_SERVICE_FIXE"] = pd.to_datetime(out["PRISE_DE_SERVICE_FIXE"], errors="coerce")
    out["START_AVANT_FENETRE"] = out["PRISE_DE_SERVICE_FIXE"].lt(min_p)

    # ---------- 2) Base pré-fenêtre par PERSONNE ----------
    if include_prewindow_base:
        from pandas.tseries.offsets import MonthBegin

        start_window = min_p                      # 1er jour de la fenêtre (ex : 2015-01-01)
        end_pre = start_window - MonthBegin(1)    # 1er jour du mois précédent (ex : 2014-12-01)

        def months_diff_inclusive(d0: pd.Timestamp, d1: pd.Timestamp) -> int:
            """Nb de mois entre d0 et d1, inclusivement, en ignorant les jours."""
            if pd.isna(d0) or pd.isna(d1) or d0 > d1:
                return 0
            return (d1.year - d0.year) * 12 + (d1.month - d0.month) + 1

        base_pre = (
            out[[matricule_col, "PRISE_DE_SERVICE_FIXE"]]
            .dropna(subset=["PRISE_DE_SERVICE_FIXE"])
            .drop_duplicates(subset=[matricule_col], keep="first")
        ).copy()

        base_pre["BASE_PRE2015_MOIS_PERS"] = base_pre["PRISE_DE_SERVICE_FIXE"].apply(
            lambda d: months_diff_inclusive(d, end_pre) if d < start_window else 0
        )

        out = out.merge(
            base_pre[[matricule_col, "BASE_PRE2015_MOIS_PERS"]],
            on=matricule_col, how="left"
        )
        out["BASE_PRE2015_MOIS_PERS"] = out["BASE_PRE2015_MOIS_PERS"].fillna(0).astype(int)
    else:
        out["BASE_PRE2015_MOIS_PERS"] = 0

    # ----------------------------------------------------------------
    # OUTIL : calendrier complet via produit cartésien (évite la perte des colonnes de clé)
    # ----------------------------------------------------------------
    def _anciennete_mensuelle(group_keys_prefix: list[str], suffix: str):
        """
        Construit un tableau mensuel complet par (group_keys_prefix) x PERIODE :
          - NB_APPA_MOIS{suffix} = nombre de lignes dans le mois
          - PRESENCE_MOIS{suffix} = 1 si NB_APPA_MOIS > 0, sinon 0
          - ANCIENNETE_MOIS{suffix} = somme cumulée des PRESENCE_MOIS par groupe
          - ANCIENNETE_AN{suffix} = ANCIENNETE_MOIS{suffix} // 12
        """
        keys = group_keys_prefix + ["PERIODE"]

        # a) Compter les apparitions (toutes lignes) par mois
        counts = (
            out.groupby(keys, dropna=False)
               .size()
               .rename(f"NB_APPA_MOIS{suffix}")
               .reset_index()
        )

        # b) Produit cartésien (groupes × full_months)
        groups = counts[group_keys_prefix].drop_duplicates()
        months_df = pd.DataFrame({"PERIODE": full_months})
        groups["_key"] = 1
        months_df["_key"] = 1
        grid = groups.merge(months_df, on="_key").drop(columns="_key")

        # c) Joindre les comptes et compléter à 0
        filled = grid.merge(counts, on=keys, how="left")
        nb_col = f"NB_APPA_MOIS{suffix}"
        filled[nb_col] = filled[nb_col].fillna(0).astype(int)

        # d) Présence binaire
        pres_col = f"PRESENCE_MOIS{suffix}"
        filled[pres_col] = (filled[nb_col] > 0).astype(int)

        # e) Ancienneté cumulée en mois (dans la fenêtre)
        anc_col = f"ANCIENNETE_MOIS{suffix}"
        filled[anc_col] = (
            filled.sort_values(group_keys_prefix + ["PERIODE"])
                  .groupby(group_keys_prefix, dropna=False)[pres_col]
                  .cumsum()
                  .astype("Int64")
        )

        # f) Années (entières)
        anc_year_col = f"ANCIENNETE_AN{suffix}"
        filled[anc_year_col] = (filled[anc_col] // 12).astype("Int64")

        return filled

    # ---------- 3) Ancienneté par ORGANISME ----------
    df_org = None
    if return_by_org and (org_col is not None) and (org_col in out.columns):
        df_org = _anciennete_mensuelle([matricule_col, org_col], "_ORG")

    # ---------- 4) Ancienneté par PERSONNE ----------
    df_pers = None
    if return_by_person:
        # Construire les comptes à partir de out (tous organismes confondus)
        # → On part de out mais on regroupe seulement par matricule et PERIODE.
        tmp = out[[matricule_col, "PERIODE"]].copy()
        tmp["_ones"] = 1
        counts_pers = (
            tmp.groupby([matricule_col, "PERIODE"], dropna=False)["_ones"]
               .sum()
               .rename("NB_APPA_MOIS_PERS")
               .reset_index()
        )

        # Produit cartésien (matricules × full_months)
        pers = counts_pers[[matricule_col]].drop_duplicates()
        months_df = pd.DataFrame({"PERIODE": full_months})
        pers["_key"] = 1
        months_df["_key"] = 1
        grid_pers = pers.merge(months_df, on="_key").drop(columns="_key")

        df_pers = grid_pers.merge(counts_pers, on=[matricule_col, "PERIODE"], how="left")
        df_pers["NB_APPA_MOIS_PERS"] = df_pers["NB_APPA_MOIS_PERS"].fillna(0).astype(int)

        df_pers["PRESENCE_MOIS_PERS"] = (df_pers["NB_APPA_MOIS_PERS"] > 0).astype(int)
        df_pers["ANCIENNETE_MOIS_PERS"] = (
            df_pers.sort_values([matricule_col, "PERIODE"])
                   .groupby([matricule_col], dropna=False)["PRESENCE_MOIS_PERS"]
                   .cumsum()
                   .astype("Int64")
        )
        df_pers["ANCIENNETE_AN_PERS"] = (df_pers["ANCIENNETE_MOIS_PERS"] // 12).astype("Int64")

    # ---------- 5) Joins sur le niveau mensuel ----------
    base_keys = [matricule_col, "PERIODE"]

    if (return_by_org and (df_org is not None)):
        out = out.merge(df_org, on=[matricule_col, org_col, "PERIODE"], how="left")

    if (return_by_person and (df_pers is not None)):
        out = out.merge(df_pers, on=base_keys, how="left")

        # Ajouter la base pré-fenêtre à l'ancienneté cumulée PERSONNE
        out["ANCIENNETE_MOIS_PERS_TOTAL"] = (
            out["BASE_PRE2015_MOIS_PERS"].astype(int) + out["ANCIENNETE_MOIS_PERS"].fillna(0).astype(int)
        ).astype("Int64")
        out["ANCIENNETE_AN_PERS_TOTAL"] = (out["ANCIENNETE_MOIS_PERS_TOTAL"] // 12).astype("Int64")

    # ---------- 6) Nettoyage colonnes optionnel ----------
    if not keep_counts:
        drop_cols = [c for c in out.columns if c.startswith("NB_APPA_MOIS")]
        out.drop(columns=drop_cols, inplace=True, errors="ignore")

    # ---------- 7) Report synthétique ----------
    rep = {
        "periode_min": min_p.date(),
        "periode_max": max_p.date(),
        "nb_matricules": out[matricule_col].nunique(),
        "exemple_pers": None
    }
    if return_by_person and ("ANCIENNETE_MOIS_PERS" in out.columns):
        rep["exemple_pers"] = (
            out[[matricule_col, "PERIODE", "NB_APPA_MOIS_PERS", "PRESENCE_MOIS_PERS",
                 "ANCIENNETE_MOIS_PERS", "BASE_PRE2015_MOIS_PERS", "ANCIENNETE_MOIS_PERS_TOTAL"]]
            .sort_values([matricule_col, "PERIODE"])
            .head(20)
        )

    return out, rep


#### CREATION DE LA VARIBALE ANCIENNTE DANS L'ORGANISME

In [12]:
import pandas as pd
import numpy as np

def build_anciennete_organisme_depuis_ps_min_collecte(
    df: pd.DataFrame,
    matricule_col: str = "MATRICULE",
    org_col: str = "ID_ORGANISME",
    collect_col: str = "DATE_COLLECTE",
    ps_col: str = "PRISE DE SERVICE",       # la colonne fournie dans ta base
    entry_col: str = "DATE_ENTREE_ORG",      # date d'entrée retenue (PS à min DATE_COLLECTE)
    months_col: str = "ANCIENNETE_ORG_MOIS",
    years_col: str = "ANCIENNETE_ORG_AN",
    inclusive: bool = True,                  # True => le mois d’entrée compte (mai→mai = 1)
    fallback_when_ps_missing: str = "min_collecte",  
    # "min_collecte" => si la PS est manquante sur la ligne la + vieille, on prend min(DATE_COLLECTE)
    # "min_ps_non_null" => sinon chercher la + vieille PRISE DE SERVICE non nulle du couple
    return_report: bool = True
):
    """
    Logique :
      1) Trouver, pour chaque couple (MATRICULE, ORGANISME), la ligne ayant la plus vieille DATE_COLLECTE.
      2) Prendre la PRISE DE SERVICE de CETTE ligne comme date d'entrée (DATE_ENTREE_ORG).
         - Si cette PS est manquante :
             * fallback_when_ps_missing == "min_ps_non_null" : on prend la plus vieille PS non nulle du couple.
             * sinon (par défaut) : on prend la plus vieille DATE_COLLECTE comme entrée.
      3) Pour chaque ligne, calculer ANCIENNETE_ORG_MOIS = écart calendaire en mois entre DATE_ENTREE_ORG et le mois de la ligne
         (mois d’entrée compté si inclusive=True).
      4) ANCIENNETE_ORG_AN = ANCIENNETE_ORG_MOIS // 12

    Remarque :
      - On n’invente aucune ligne : calcul au niveau des lignes existantes.
    """
    out = df.copy()

    # -- sécuriser dates --
    out[collect_col] = pd.to_datetime(out[collect_col], errors="coerce")
    out[ps_col] = pd.to_datetime(out[ps_col], errors="coerce")
    if out[collect_col].isna().all():
        raise ValueError(f"Toutes les valeurs de '{collect_col}' sont manquantes ou invalides.")

    # PERIODE = mois civil (début de mois)
    out["PERIODE"] = out[collect_col].dt.to_period("M").dt.to_timestamp(how="start")

    # 1) identifier, par couple, la ligne à min DATE_COLLECTE
    base = (
        out[[matricule_col, org_col, collect_col, ps_col]]
        .sort_values([matricule_col, org_col, collect_col], ascending=[True, True, True])
        .groupby([matricule_col, org_col], dropna=False, as_index=False)
        .first()   # conserve la ligne à plus vieille collecte
        .rename(columns={ps_col: "_PS_SUR_MINCOLLECTE", collect_col: "_MIN_COLLECTE"})
    )

    # 2) fallback si PS manquante sur la ligne min_collecte
    if fallback_when_ps_missing == "min_ps_non_null":
        # chercher la plus vieille PS non nulle du couple
        min_ps = (
            out[[matricule_col, org_col, ps_col]]
            .dropna(subset=[ps_col])
            .sort_values([matricule_col, org_col, ps_col])
            .groupby([matricule_col, org_col], dropna=False, as_index=False)
            .first()
            .rename(columns={ps_col: "_MIN_PS_NON_NULL"})
        )
        base = base.merge(min_ps, on=[matricule_col, org_col], how="left")

        # règle : PS = _PS_SUR_MINCOLLECTE si dispo, sinon _MIN_PS_NON_NULL, sinon _MIN_COLLECTE
        base[entry_col] = np.where(
            base["_PS_SUR_MINCOLLECTE"].notna(),
            base["_PS_SUR_MINCOLLECTE"],
            np.where(base["_MIN_PS_NON_NULL"].notna(), base["_MIN_PS_NON_NULL"], base["_MIN_COLLECTE"])
        )
    else:
        # règle par défaut : PS = _PS_SUR_MINCOLLECTE si dispo, sinon min collecte
        base[entry_col] = np.where(
            base["_PS_SUR_MINCOLLECTE"].notna(),
            base["_PS_SUR_MINCOLLECTE"],
            base["_MIN_COLLECTE"]
        )

    base = base[[matricule_col, org_col, entry_col]]

    # 3) joindre l'entrée au panel
    out = out.drop(columns=[entry_col], errors="ignore")
    out = out.merge(base, on=[matricule_col, org_col], how="left")
    out[entry_col] = pd.to_datetime(out[entry_col], errors="coerce")

    # 4) calcul des mois inclusifs
    def _months_diff_inclusive(d0, d1, inclusive=True):
        if pd.isna(d0) or pd.isna(d1):
            return np.nan
        m0 = pd.Timestamp(d0).to_period("M")
        m1 = pd.Timestamp(d1).to_period("M")
        if m1 < m0:
            return 0
        base = (m1.year - m0.year) * 12 + (m1.month - m0.month)
        return base + 1 if inclusive else base

    out[months_col] = [
        _months_diff_inclusive(de, pe, inclusive) for de, pe in zip(out[entry_col], out["PERIODE"])
    ]
    out[months_col] = pd.to_numeric(out[months_col], errors="coerce").astype("Int64")

    # 5) années
    out[years_col] = (out[months_col] // 12).astype("Int64")

    if return_report:
        rep = {
            "nb_couples_personne_organisme": int(base.shape[0]),
            "extrait": out[[matricule_col, org_col, "PERIODE", entry_col, months_col, years_col]]
                        .sort_values([matricule_col, org_col, "PERIODE"])
                        .head(20)
        }
        return out, rep

    return out


# CHARGEMENT DE LA BASE DE TRAVAIL

In [13]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"

# object_key 
object_key = "Solde/panel_solde_df_1_code.parquet"  # Chemin dans le bucket

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Afficher un aperçu
panel_solde_df.head()

,MATRICULE||CODE_ORGANISME,MATRICULE,NOM,DATE NAISSANCE,SEXE,SITUATION MATRIMONIALE,NOMBRE ENFANT,LIEU AFFECTATION,SERVICE,ORGANISME,...,SERVICE_NORM,CODE_SERVICE,ORGANISME_NORM,CODE_ORGANISME,CLASSE/ECHELON_NORM,CODE_CLASSE/ECHELON,EMPLOI_NORM,CODE_EMPLOI,SITUATION_NORM,CODE_SITUATION
0,011242X15,011242X,DOSSO MEGBO,1939-01-01,Masculin,Marié,0.0,Seguela,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
1,012856Q15,012856Q,KHISSY BEYNIOUAH FULBERT,1939-01-01,Masculin,Célibataire,0.0,Bouaké,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
2,013454N15,013454N,VAHA TOMAS GNOMBLEI HENRI,1924-01-01,Masculin,Marié,0.0,Guiglo,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
3,027861L25,027861L,CAPET YATO MATHIEU,1943-03-01,Masculin,Marié,0.0,Abidjan,A affecter,"Min. Affaires Etrangères, de l'Intégration Afr...",...,a affecter,1800,"min affaires etrangeres, de l integration afri...",25,,<NA>,,<NA>,en activite,10
4,039659M38,039659M,COULIBALY YOUSSOUF,1945-12-01,Masculin,Marié,2.0,Abidjan,Dir Personnel Police Nationale,Min. de l'Intérieur et de la Sécurité,...,dir personnel police nationale,3813,min de l interieur et de la securite,38,commissaire divis 1er ech,LW,commissaire de police,P1ZC,retraite pour limite age,60


In [14]:
df = panel_solde_df.copy()

# 1) Inspection rapide des types (optionnel)
print(df[["PERIODE", "DATE_COLLECTE"]].dtypes)

PERIODE           int64
DATE_COLLECTE    object
dtype: object


In [15]:
import pandas as pd
import numpy as np

df = panel_solde_df.copy()

# 1) Parser PERIODE (int YYYYMM) -> Period[M]
p_str = pd.to_numeric(df["PERIODE"], errors="coerce").astype("Int64").astype(str).str.zfill(6)
year_p = pd.to_numeric(p_str.str[:4], errors="coerce")
month_p = pd.to_numeric(p_str.str[4:6], errors="coerce")
valid_p = month_p.between(1, 12)
periode_M = pd.Series(pd.NaT, index=df.index, dtype="period[M]")
periode_M[valid_p] = pd.PeriodIndex(year=year_p[valid_p], month=month_p[valid_p], freq="M")

# 2) Parser DATE_COLLECTE (str) -> datetime -> Period[M]
dc = pd.to_datetime(df["DATE_COLLECTE"], errors="coerce") \
       .fillna(pd.to_datetime(df["DATE_COLLECTE"], errors="coerce", dayfirst=True))
date_M = dc.dt.to_period("M")

df["_PERIODE_M"] = periode_M
df["_DATE_COLLECTE_M"] = date_M

# 3) Comparaison au mois
same_month = df["_PERIODE_M"].eq(df["_DATE_COLLECTE_M"])
both_na = df["_PERIODE_M"].isna() & df["_DATE_COLLECTE_M"].isna()

print("=== Résumé ===")
print(f"Total: {len(df):,}")
print(f"Égalité mois: {same_month.sum():,} ({same_month.mean():.1%})")
print(f"Incohérences: {(~same_month & ~both_na).sum():,} ({(~same_month & ~both_na).mean():.1%})")

# 4) Écart en MOIS sans TypeError (méthode année*12 + mois)
valid = df["_PERIODE_M"].notna() & df["_DATE_COLLECTE_M"].notna()

month_diff = pd.Series(pd.NA, index=df.index, dtype="Int64")
y_p = df.loc[valid, "_PERIODE_M"].dt.year
m_p = df.loc[valid, "_PERIODE_M"].dt.month
y_d = df.loc[valid, "_DATE_COLLECTE_M"].dt.year
m_d = df.loc[valid, "_DATE_COLLECTE_M"].dt.month

month_diff.loc[valid] = (y_d*12 + m_d) - (y_p*12 + m_p)
df["_ECART_MOIS"] = month_diff

print("\n=== Distribution des écarts (mois) ===")
print(month_diff.value_counts(dropna=True).sort_index())

# 5) Écart en jours (DATE_COLLECTE - 1er jour de PERIODE), idem avec masque
ecart_jours = pd.Series(pd.NA, index=df.index, dtype="Int64")
periode_as_date = df["_PERIODE_M"].dt.to_timestamp(how="start")
ecart_jours.loc[valid] = (dc.loc[valid] - periode_as_date.loc[valid]).dt.days.astype("Int64")
df["_ECART_JOURS"] = ecart_jours

print("\n=== Écart en jours (stats) ===")
print(ecart_jours.dropna().describe())

# 6) Exemples d’incohérences
mismatch = df.loc[~same_month & ~both_na, ["PERIODE", "DATE_COLLECTE", "_PERIODE_M", "_DATE_COLLECTE_M", "_ECART_MOIS", "_ECART_JOURS"]]
print("\n=== Exemples d'incohérences (5) ===")
print(mismatch.head(5))


/tmp/ipykernel_23/58105453.py:12: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  periode_M[valid_p] = pd.PeriodIndex(year=year_p[valid_p], month=month_p[valid_p], freq="M")


=== Résumé ===
Total: 15,627,963
Égalité mois: 0 (0.0%)
Incohérences: 15,627,963 (100.0%)

=== Distribution des écarts (mois) ===
Series([], Name: count, dtype: Int64)

=== Écart en jours (stats) ===
count     0.0
mean     <NA>
std      <NA>
min      <NA>
25%      <NA>
50%      <NA>
75%      <NA>
max      <NA>
dtype: Float64

=== Exemples d'incohérences (5) ===
   PERIODE DATE_COLLECTE _PERIODE_M _DATE_COLLECTE_M  _ECART_MOIS  \
0    12015    2015-01-01        NaT          2015-01         <NA>   
1    12015    2015-01-01        NaT          2015-01         <NA>   
2    12015    2015-01-01        NaT          2015-01         <NA>   
3    12015    2015-01-01        NaT          2015-01         <NA>   
4    12015    2015-01-01        NaT          2015-01         <NA>   

   _ECART_JOURS  
0          <NA>  
1          <NA>  
2          <NA>  
3          <NA>  
4          <NA>  


In [16]:
import pandas as pd
path = r"C:\Users\a_coulibaly\OneDrive - Metrics & Decisions\INS\CAE\Banque de données travailleurs\Data\Solde\panel_solde_df_1_code.parquet"

panel_solde_df = pd.read_parquet(path, engine="fastparquet")  # <-- alternative à pyarrow

# Afficher un aperçu
panel_solde_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\a_coulibaly\\OneDrive - Metrics & Decisions\\INS\\CAE\\Banque de données travailleurs\\Data\\Solde\\panel_solde_df_1_code.parquet'

In [ ]:
print (panel_solde_df.columns)
print(f"Nous avons {len(panel_solde_df)} observations")

# AFFICHAGE DES DIFFERENTES VARIABLES CREES 

## CATEGORIE 

In [17]:
# Appliquer la fonction de création des catégories 1 et 2
panel_solde_df = build_categories(panel_solde_df, grade_col="GRADE", grade2_col="GRADE 2")

# Aperçu
panel_solde_df[["GRADE","CATEGORIE_1","GRADE 2","CATEGORIE_2"]].head()

,GRADE,CATEGORIE_1,GRADE 2,CATEGORIE_2
0,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
1,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
2,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
3,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
4,Fonctionnaire Catégorie A7,A,A7,A


In [19]:
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'CODE_ORGANISME', 'CLASSE/ECHELON_NORM', 'CODE_CLASSE/ECHELON',
       'EMPLOI_NORM', 'CODE_EMPLOI', 'SITUATION_NORM', 'CODE_SITUATION',
       'CATEGORIE_1', 'CATEGORIE_2'],
      dtype='object')

In [25]:
print(panel_solde_df['CODE_ORGANISME'].count())


15622889


### Extraction des modalités des variables GRADE et GRADE 2

In [26]:
# Étape 1 : Nettoyer les données
unique_grades = panel_solde_df[["GRADE", "CATEGORIE_1"]].dropna()  # Supprime les NaN
unique_grades = unique_grades[(unique_grades["GRADE"] != "") & (unique_grades["CATEGORIE_1"] != "")]  # Supprime les vides

# Étape 2 : Extraire les lignes uniques
unique_grades = unique_grades.drop_duplicates().reset_index(drop=True)

# Étape 3 : Sauvegarde dans Excel
unique_grades.to_excel("unique_grades.xlsx", index=False)


In [27]:
# Étape 1 : Nettoyer les données
unique_grades = panel_solde_df[["GRADE 2", "CATEGORIE_2"]].dropna()  # Supprime les NaN
unique_grades = unique_grades[(unique_grades["GRADE 2"] != "") & (unique_grades["CATEGORIE_2"] != "")]  # Supprime les vides

# Étape 2 : Extraire les lignes uniques
unique_grades = unique_grades.drop_duplicates().reset_index(drop=True)

# Étape 3 : Sauvegarde dans Excel
unique_grades.to_excel("unique_grades_2.xlsx", index=False)

In [28]:
# 📊 Créer un tableau croisé des effectifs
tableau_croise_effectifs = pd.crosstab(
    panel_solde_df["CATEGORIE_1"],   # Variable en ligne
    panel_solde_df["CATEGORIE_2"],   # Variable en colonne
    margins=True,        # Ajoute les totaux
    margins_name="Total" # Nom de la ligne/colonne total
)
tableau_croise_effectifs

CATEGORIE_2,Non Fonctionnaire,A,B,C,D,Total
CATEGORIE_1,,,,,,
Non Fonctionnaire,320356,43,0,0,0,320399
A,0,4797789,0,0,0,4797789
B,0,0,6119667,0,0,6119667
C,0,0,0,3961427,0,3961427
D,0,0,0,0,428681,428681
Total,320356,4797832,6119667,3961427,428681,15627963


In [29]:
print(((panel_solde_df["CATEGORIE_1"].isna()) | (panel_solde_df["CATEGORIE_1"].str.strip() == "")).sum())
print(((panel_solde_df["CATEGORIE_2"].isna()) | (panel_solde_df["CATEGORIE_2"].str.strip() == "")).sum())

0
0


In [30]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["EMPLOI_NORM"].value_counts()
print(tab_statut)

EMPLOI_NORM
                         36
medecin generaliste       5
inspecteur des impots     2
Name: count, dtype: int64


## STATUT

In [31]:
# Appliquer la fonction de création de la variable statut
panel_solde_df, rep = build_statut_from_cats(panel_solde_df)
rep

{'repartition_statut': STATUT
 Non Fonctionnaire      315125
 Fonctionnaire        15307347
 Cas particulier          5491
 Name: count, dtype: int64}

In [32]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["STATUT"].value_counts()
print(tab_statut)

STATUT
Cas particulier      43
Non Fonctionnaire     0
Fonctionnaire         0
Name: count, dtype: int64


In [33]:
# Tabulation simple sur STATUT
tab_statut = panel_solde_df["STATUT"].value_counts(dropna=False).sort_index()

# Total lignes
total_lignes = tab_statut.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut": tab_statut.index,
    "Effectif": tab_statut.values,
    "Proportion (%)": (tab_statut.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

,Statut,Effectif,Proportion (%)
0,Non Fonctionnaire,315125,2.02
1,Fonctionnaire,15307347,97.95
2,Cas particulier,5491,0.04
3,Total,15627963,100.01


## STATUT DEFINITIF

In [34]:
# Appliquer la fonction de création du statut définitif en s'appuyant sur la période MMYYY
panel_solde_df, rep = build_statut_def_periode(panel_solde_df, return_report=True)
rep

{'Statut définifitif': Statut_def_mois
 Non Fonctionnaire      231219
 Cas particulier          4964
 Fonctionnaire        15391780
 Name: count, dtype: int64}

In [35]:
# Récupérer la répartition des lignes par Statut_def
tab_Statut_def = rep["Statut définifitif"]

# Total lignes
total_lignes = tab_Statut_def.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut Définitif": tab_Statut_def.index,
    "Effectif": tab_Statut_def.values,
    "Proportion (%)": (tab_Statut_def.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut Définitif": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

,Statut Définitif,Effectif,Proportion (%)
0,Non Fonctionnaire,231219,1.48
1,Cas particulier,4964,0.03
2,Fonctionnaire,15391780,98.49
3,Total,15627963,100.00


In [36]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["Statut_def_mois"].value_counts()
print(tab_statut)

Statut_def_mois
Cas particulier      43
Non Fonctionnaire     0
Fonctionnaire         0
Name: count, dtype: int64


In [37]:
# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_mois"] == "Cas particulier"]

# Vérifier si un même matricule apparaît plusieurs fois dans la même période
doublons = cas_particuliers_df[cas_particuliers_df.duplicated(subset=["MATRICULE", "PERIODE"], keep=False)]

if doublons.empty:
    print("✅ Chaque matricule est unique par période.")
else:
    print(f"⚠️ {len(doublons)} doublons trouvés (mêmes matricules dans la même période).")
    print(doublons[["MATRICULE", "PERIODE"]].sort_values(by=["MATRICULE", "PERIODE"]))

⚠️ 2 doublons trouvés (mêmes matricules dans la même période).
        MATRICULE  PERIODE
9228788   089081R   102018
9228789   089081R   102018


In [38]:
# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_mois"] == "Cas particulier"]

# Trier par la colonne "MATRICULE||CODE_ORGANISME"
cas_particuliers_df = cas_particuliers_df.sort_values(by="MATRICULE")

# Chemin et nom du fichier Excel de sortie
output_file = "cas_particuliers_def_periode.xlsx"

# Exporter avec index
cas_particuliers_df[["MATRICULE", "MATRICULE||CODE_ORGANISME", "EMPLOI_NORM", "CATEGORIE_1", "CATEGORIE_2", "PERIODE"]].to_excel(output_file, index=False)

print(f"Fichier exporté : {output_file} avec {len(cas_particuliers_df)} observations")

Fichier exporté : cas_particuliers_def_periode.xlsx avec 4964 observations


In [31]:
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'CODE_ORGANISME', 'CLASSE/ECHELON_NORM', 'CODE_CLASSE/ECHELON',
       'EMPLOI_NORM', 'CODE_EMPLOI', 'SITUATION_NORM', 'CODE_SITUATION',
       'CATEGORIE_1', 'CATEGORIE_2', 'STATUT', 'Statut_def_mois'],
      dtype='object')

In [39]:
# Appliquer la fonction de création du statut définitif en s'appuyant sur l'année
panel_solde_df, rep = build_statut_def_annee(panel_solde_df, return_report=True)
rep

{'Statut_def_distribution': Statut_def_annee
 Non Fonctionnaire      230930
 Cas particulier          3469
 Fonctionnaire        15393564
 Name: count, dtype: int64}

In [40]:
# Récupérer la répartition des lignes par Statut_def
tab_Statut_def = rep["Statut_def_distribution"]

# Total lignes
total_lignes = tab_Statut_def.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut Définitif": tab_Statut_def.index,
    "Effectif": tab_Statut_def.values,
    "Proportion (%)": (tab_Statut_def.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut Définitif": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

,Statut Définitif,Effectif,Proportion (%)
0,Non Fonctionnaire,230930,1.48
1,Cas particulier,3469,0.02
2,Fonctionnaire,15393564,98.50
3,Total,15627963,100.00


In [41]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A") & 
    (panel_solde_df["EMPLOI_NORM"] != "") 

]
tab_statut = filtre["Statut_def_annee"].value_counts()
print(tab_statut)

Statut_def_annee
Fonctionnaire        7
Non Fonctionnaire    0
Cas particulier      0
Name: count, dtype: int64


In [42]:
# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_annee"] == "Cas particulier"]

# Trier par la colonne "MATRICULE||CODE_ORGANISME"
cas_particuliers_df = cas_particuliers_df.sort_values(by="MATRICULE")

# Chemin et nom du fichier Excel de sortie
output_file = "cas_particuliers_def_annee.xlsx"

# Exporter avec index
cas_particuliers_df[["MATRICULE", "MATRICULE||CODE_ORGANISME", "EMPLOI_NORM", "CATEGORIE_1", "CATEGORIE_2", "ANNEE"]].to_excel(output_file, index=False)

print(f"Fichier exporté : {output_file} avec {len(cas_particuliers_df)} observations")

Fichier exporté : cas_particuliers_def_annee.xlsx avec 3469 observations


## SEXE

In [43]:
panel_solde_df, report = imputer_sexe(
    panel_solde_df,
    sex_col="SEXE",
    collect_col="DATE_COLLECTE",
    sex_valid_col="SEXE_IMPUTE",
    debug=True
)
report

=== Statistiques avant imputation ===


,Effectif
SEXE,
Féminin,4683169
Masculin,10944644
None,150



=== Statistiques après imputation ===


,Effectif
SEXE_IMPUTE,
Féminin,4683169
Masculin,10944794



=== Pourcentages après imputation ===


,Pourcentage (%)
SEXE_IMPUTE,
Féminin,29.966599
Masculin,70.033401



=== Tableau croisé avant/après ===


SEXE_IMPUTE,Féminin,Masculin,Total
SEXE,,,
Féminin,4683169,0,4683169.0
Masculin,0,10944644,10944644.0
NaN,0,150,NaN
Total,4683169,10944794,15627963.0


{'tab_sexe_avant': SEXE
 Féminin      4683169
 Masculin    10944644
 None             150
 Name: count, dtype: int64,
 'tab_sexe_apres': SEXE_IMPUTE
 Féminin      4683169
 Masculin    10944794
 Name: count, dtype: int64,
 'tab_sexe_apres_pct': SEXE_IMPUTE
 Féminin     29.966599
 Masculin    70.033401
 Name: proportion, dtype: float64,
 'crosstab_sexe': SEXE_IMPUTE  Féminin  Masculin       Total
 SEXE                                      
 Féminin      4683169         0   4683169.0
 Masculin           0  10944644  10944644.0
 NaN                0       150         NaN
 Total        4683169  10944794  15627963.0}

In [44]:
variation, details, panel_solde_df, stats = verifier_variation_sexe(
    panel_solde_df, id_col="MATRICULE", sex_col="SEXE_IMPUTE", collect_col="DATE_COLLECTE"
)

# Voir les stats après mise à jour
stats


Nombre d'individus dont le sexe varie dans le temps : 2523


{'tab_avant': SEXE_IMPUTE
 Féminin      4683169
 Masculin    10944794
 Name: count, dtype: int64,
 'tab_apres': SEXE_IMPUTE
 Féminin      4684666
 Masculin    10943297
 Name: count, dtype: int64,
 'tab_apres_pct': SEXE_IMPUTE
 Féminin     29.976178
 Masculin    70.023822
 Name: proportion, dtype: float64}

In [38]:
print(panel_solde_df.columns)

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'CODE_ORGANISME', 'CLASSE/ECHELON_NORM', 'CODE_CLASSE/ECHELON',
       'EMPLOI_NORM', 'CODE_EMPLOI', 'SITUATION_NORM', 'CODE_SITUATION',
       'CATEGORIE_1', 'CATEGORIE_2', 'STATUT', 'Statut_def_mois', 'ANNEE',
       'Statut_def_annee', 'ANNEE_COLLECTE', 'MOIS_COLLECTE', 'SEXE_IMPUTE'],
      dtype='object')


## AGE 

### AGE Valide

In [45]:
# 0) S'assurer qu'on a SEXE_IMPUTE
if "SEXE_IMPUTE" not in panel_solde_df.columns:
    panel_solde_df, _ = imputer_sexe(
        panel_solde_df,
        sex_col="SEXE",
        collect_col="DATE_COLLECTE",
        sex_valid_col="SEXE_IMPUTE",
        debug=False
    )

# 1) Calcul des âges (build_age_valide ne renvoie que le df)
panel_solde_df = build_age_valide(
    panel_solde_df,
    matricule_col="MATRICULE",
    birth_col="DATE NAISSANCE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    age_min=18, age_max=70,
    do_impute_age=True
)

# 2) Reconstruire les tableaux "rep" à partir du df
tab_sexe_avant      = panel_solde_df["SEXE"].value_counts(dropna=False).sort_index()
tab_sexe_avant_pct  = panel_solde_df["SEXE"].value_counts(normalize=True, dropna=False).sort_index() * 100
tab_sexe_apres      = panel_solde_df["SEXE_IMPUTE"].value_counts(dropna=False).sort_index()
tab_sexe_apres_pct  = panel_solde_df["SEXE_IMPUTE"].value_counts(normalize=True, dropna=False).sort_index() * 100
crosstab_sexe       = pd.crosstab(panel_solde_df["SEXE"], panel_solde_df["SEXE_IMPUTE"], dropna=False, margins=True, margins_name="Total")

rep = {
    "tab_sexe_avant": tab_sexe_avant,
    "tab_sexe_avant_pct": tab_sexe_avant_pct,
    "tab_sexe_apres": tab_sexe_apres,
    "tab_sexe_apres_pct": tab_sexe_apres_pct,
    "crosstab_sexe": crosstab_sexe
}

# 3) Afficher les tableaux
print(pd.DataFrame({"Effectif": rep["tab_sexe_avant"], "Pourcentage (%)": rep["tab_sexe_avant_pct"]}))
print(pd.DataFrame({"Effectif": rep["tab_sexe_apres"],  "Pourcentage (%)": rep["tab_sexe_apres_pct"]}))
print(rep["crosstab_sexe"])

# 4) Aperçu des colonnes ajoutées
cols_preview = [
    "MATRICULE","DATE_NAISSANCE_CORRIGEE",
    "ANNEE_COLLECTE","MOIS_COLLECTE",
    "SEXE","SEXE_IMPUTE",
    "AGE","AGE_VALIDE","AGE_IMPUTE"
]
print(panel_solde_df[cols_preview].head())


          Effectif  Pourcentage (%)
SEXE                               
Féminin    4683169        29.966599
Masculin  10944644        70.032441
None           150         0.000960
             Effectif  Pourcentage (%)
SEXE_IMPUTE                           
Féminin       4684666        29.976178
Masculin     10943297        70.023822
SEXE_IMPUTE  Féminin  Masculin       Total
SEXE                                      
Féminin      4678407      4762   4683169.0
Masculin        6133  10938511  10944644.0
NaN              126        24         NaN
Total        4684666  10943297  15627963.0
  MATRICULE DATE_NAISSANCE_CORRIGEE  ANNEE_COLLECTE  MOIS_COLLECTE      SEXE  \
0   011242X              1939-01-01            2015              1  Masculin   
1   012856Q              1939-01-01            2015              1  Masculin   
2   013454N              1924-01-01            2015              1  Masculin   
3   027861L              1943-03-01            2015              1  Masculin   
4   03

#### AGE DE PRISE DE SERVICE 

In [46]:
panel_solde_df, rep_srv = build_age_service(
    panel_solde_df,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",            # sexe imputé déjà calculé
    year_collect_col="ANNEE_COLLECTE",
    month_collect_col="MOIS_COLLECTE",
    recompute_corrected=True,         # recalcule la prise de service “corrigée”
    age_min=18, age_max=40,           # bornage de l’âge au service
    days_per_year=365,
    return_tables=True
)


In [47]:
# Aperçu des nouvelles colonnes
panel_solde_df[[
    "MATRICULE", "PRISE_DE_SERVICE_CORRIGEE", "DATE_COLLECTE",
    "AGE_IMPUTE", "AGE_AU_SERVICE", "AGE_AU_SERVICE_VALIDE", "AGE_AU_SERVICE_IMPUTE"
]].head()

# Tables de synthèse fournies
print(rep_srv["tab_age_service_valide"])
print(rep_srv["tab_age_service_impute"])


                       Effectif  Pourcentage (%)
AGE_AU_SERVICE_VALIDE                           
18.0                      58002             0.37
19.0                     131233             0.84
20.0                     222657             1.42
21.0                     356505             2.28
22.0                     523426             3.35
23.0                     758065             4.85
24.0                    1005464             6.43
25.0                    1193650             7.64
26.0                    1328606              8.5
27.0                    1346349             8.61
28.0                    1273666             8.15
29.0                    1166479             7.46
30.0                    1041962             6.67
31.0                     928126             5.94
32.0                     815669             5.22
33.0                     701753             4.49
34.0                     614484             3.93
35.0                     520605             3.33
36.0                

#### ANCIENNETE

#### Ancienneté dans la fonction

In [48]:
panel_solde_df_mois, rep_mois = build_anciennete_mensuelle(
    panel_solde_df,
    matricule_col="MATRICULE",
    org_col="CODE_ORGANISME",
    collect_col="DATE_COLLECTE",
    start_service_col="PRISE_DE_SERVICE_CORRIGEE",
    min_period="2015-01",
    max_period="2020-12",
    return_by_org=True,
    return_by_person=True,
    keep_counts=True,
    include_prewindow_base=True
)


In [49]:
cols_priorite = [
    "MATRICULE","ID_ORGANISME","PERIODE",
    "NB_APPA_MOIS_ORG","PRESENCE_MOIS_ORG","ANCIENNETE_MOIS_ORG","ANCIENNETE_AN_ORG",
    "NB_APPA_MOIS_PERS","PRESENCE_MOIS_PERS","ANCIENNETE_MOIS_PERS",
    "BASE_PRE2015_MOIS_PERS","ANCIENNETE_MOIS_PERS_TOTAL","ANCIENNETE_AN_PERS_TOTAL",
    "PRISE_DE_SERVICE_FIXE","START_AVANT_FENETRE"
]
cols_show = [c for c in cols_priorite if c in panel_solde_df_mois.columns]
print("\nAperçu colonnes clés :")
display(panel_solde_df_mois[cols_show].sort_values([c for c in ["MATRICULE","PERIODE"] if c in cols_show]).head(30))


Aperçu colonnes clés :


,MATRICULE,PERIODE,NB_APPA_MOIS_ORG,PRESENCE_MOIS_ORG,ANCIENNETE_MOIS_ORG,ANCIENNETE_AN_ORG,NB_APPA_MOIS_PERS,PRESENCE_MOIS_PERS,ANCIENNETE_MOIS_PERS,BASE_PRE2015_MOIS_PERS,ANCIENNETE_MOIS_PERS_TOTAL,ANCIENNETE_AN_PERS_TOTAL,PRISE_DE_SERVICE_FIXE,START_AVANT_FENETRE
0,011242X,2015-01-01,1,1,1,0,1,1,1,552,553,46,1969-01-01,True
184890,011242X,2015-02-01,1,1,2,0,1,1,2,552,554,46,1969-01-01,True
370226,011242X,2015-03-01,1,1,3,0,1,1,3,552,555,46,1969-01-01,True
556520,011242X,2015-04-01,1,1,4,0,1,1,4,552,556,46,1969-01-01,True
744494,011242X,2015-05-01,1,1,5,0,1,1,5,552,557,46,1969-01-01,True
934474,011242X,2015-06-01,1,1,6,0,1,1,6,552,558,46,1969-01-01,True
1126818,011242X,2015-07-01,1,1,7,0,1,1,7,552,559,46,1969-01-01,True
1319932,011242X,2015-08-01,1,1,8,0,1,1,8,552,560,46,1969-01-01,True
1514281,011242X,2015-09-01,1,1,9,0,1,1,9,552,561,46,1969-01-01,True
1709249,011242X,2015-10-01,1,1,10,0,1,1,10,552,562,46,1969-01-01,True


#### Ancienneté dans l'organisme

In [52]:
panel_solde_df, rep = build_anciennete_organisme_depuis_ps_min_collecte(
    panel_solde_df_mois,
    matricule_col="MATRICULE",
    org_col="CODE_ORGANISME",
    collect_col="DATE_COLLECTE",
    ps_col="PRISE DE SERVICE",
    entry_col="DATE_ENTREE_ORG",
    months_col="ANCIENNETE_ORG_MOIS",
    years_col="ANCIENNETE_ORG_AN",
    inclusive=True,
    fallback_when_ps_missing="min_collecte",  # ou "min_ps_non_null"
    return_report=True
)

display(rep["extrait"].head(15))


,MATRICULE,CODE_ORGANISME,PERIODE,DATE_ENTREE_ORG,ANCIENNETE_ORG_MOIS,ANCIENNETE_ORG_AN
0,011242X,15,2015-01-01,1969-01-01,553,46
184890,011242X,15,2015-02-01,1969-01-01,554,46
370226,011242X,15,2015-03-01,1969-01-01,555,46
556520,011242X,15,2015-04-01,1969-01-01,556,46
744494,011242X,15,2015-05-01,1969-01-01,557,46
934474,011242X,15,2015-06-01,1969-01-01,558,46
1126818,011242X,15,2015-07-01,1969-01-01,559,46
1319932,011242X,15,2015-08-01,1969-01-01,560,46
1514281,011242X,15,2015-09-01,1969-01-01,561,46
1709249,011242X,15,2015-10-01,1969-01-01,562,46
